In [1]:
!pip -q install pandas nltk spacy

import os, json, re, string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy

In [2]:
nltk.download("punkt")
nltk.download("stopwords")

!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 16.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
RAW_PATH = "gnews_articles_raw.jsonl"  # change if stored in a folder
OUT_DIR = "phase2_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RAW_PATH

'gnews_articles_raw.jsonl'

In [4]:
rows = []
with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df_raw = pd.DataFrame(rows)
print("Loaded rows:", len(df_raw))
print("Columns:", list(df_raw.columns))
df_raw.head(5)

Loaded rows: 110
Columns: ['kind', 'id', 'title', 'description', 'content', 'url', 'image', 'publishedAt', 'lang', 'source']


,kind,id,title,description,content,url,image,publishedAt,lang,source
0,gnews_article,5266cd2df1f16d65ce7437f027af9c70,Moderna shares drop after FDA refuses to file ...,Moderna’s mRNA-1010 flu vaccine application ge...,Moderna shares drop after FDA refuses to file ...,https://seekingalpha.com/news/4550005-moderna-...,https://static.seekingalpha.com/cdn/s3/uploads...,2026-02-11T07:54:39Z,en,"{'id': 'a6d401909ca644c4ffb332486c9cfaea', 'na..."
1,gnews_article,3a0666be7320f0c842d36385e1bb03cc,FDA refuses to review Moderna’s mRNA flu vacci...,FDA refuses Moderna mRNA flu vaccine applicati...,NEWYou can now listen to Fox News articles!\nT...,https://www.foxnews.com/health/fda-refuses-rev...,https://static.foxnews.com/foxnews.com/content...,2026-02-11T07:20:19Z,en,"{'id': '8da923084554b2821233d6c5c5f02c97', 'na..."
2,gnews_article,9b5bd03f3a5673010493934383d90e41,Moderna says FDA refuses its application for n...,The Food and Drug Administration is refusing t...,The Food and Drug Administration is refusing t...,https://www.cbsnews.com/news/moderna-says-fda-...,https://assets3.cbsnewsstatic.com/hub/i/r/2022...,2026-02-11T04:28:28Z,en,"{'id': '3a3847d645a619320aa3b3484581e395', 'na..."
3,gnews_article,43e92db65ff2615e4430533488062d0a,Moderna says FDA refuses its application for n...,Moderna faces FDA refusal for its mRNA flu vac...,FDA under Robert F. Kennedy Jr. refused to con...,https://www.cnbctv18.com/lifestyle/healthcare/...,https://images.cnbctv18.com/uploads/2026/02/mo...,2026-02-11T03:43:35Z,en,"{'id': '4bf882bd9fc07a7fc3ab723f04e4f261', 'na..."
4,gnews_article,fb6fb5b794937bad01110b6b9ee939a6,Moderna says FDA refuses its application for n...,The decision follows a pattern of policy shift...,"By LAURAN NEERGAARD and MATTHEW PERRONE, The A...",https://www.pennlive.com/health/2026/02/kenned...,https://www.pennlive.com/resizer/v2/4ZNKQF2FDJ...,2026-02-11T02:55:05Z,en,"{'id': '016c9538dd6aa60fb4f29fca8674d6b7', 'na..."


In [5]:
df = df_raw.copy()

# Extract source name/url if source is a dict
if "source" in df.columns:
    df["source_name"] = df["source"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)
    df["source_url"] = df["source"].apply(lambda x: x.get("url") if isinstance(x, dict) else None)
else:
    df["source_name"] = None
    df["source_url"] = None

df[["source_name", "source_url"]].head(5)

,source_name,source_url
0,Seeking Alpha,https://seekingalpha.com
1,Fox News,https://www.foxnews.com
2,CBS News,https://www.cbsnews.com
3,CNBC TV18,https://www.cnbctv18.com
4,Mechanicsburg Patriot News,https://www.pennlive.com


In [6]:
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))

if "url" in df.columns:
    print("\nUnique URLs:", df["url"].nunique())

Missing values per column:
kind           0
id             0
title          0
description    0
content        0
url            0
image          0
publishedAt    0
lang           0
source         0
source_name    0
source_url     0
dtype: int64

Unique URLs: 110


In [7]:
url_pattern = re.compile(r"https?://\S+|www\.\S+")

def clean_text_safe(s):
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    s = url_pattern.sub("", s)          # remove URLs inside text
    s = re.sub(r"\s+", " ", s).strip()  # normalize whitespace
    return s

def combine_text(row):
    parts = []
    for col in ["title", "description", "content"]:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return " ".join(parts).strip()

df["text_full"] = df.apply(combine_text, axis=1)
df["text_clean"] = df["text_full"].apply(clean_text_safe)

df[["title", "text_full", "text_clean"]].head(5)

,title,text_full,text_clean
0,Moderna shares drop after FDA refuses to file ...,Moderna shares drop after FDA refuses to file ...,Moderna shares drop after FDA refuses to file ...
1,FDA refuses to review Moderna’s mRNA flu vacci...,FDA refuses to review Moderna’s mRNA flu vacci...,FDA refuses to review Moderna’s mRNA flu vacci...
2,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...
3,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...
4,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...


In [8]:
# Drop duplicates by URL
if "url" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["url"])
    print("Dropped duplicates by url:", before - len(df))

# Drop duplicates by id
if "id" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["id"])
    print("Dropped duplicates by id:", before - len(df))

# Remove the dict column before exact duplicates
if "source" in df.columns:
    df = df.drop(columns=["source"])

# Remove exact duplicates safely
before = len(df)
df = df.drop_duplicates()
print("Dropped exact duplicate rows:", before - len(df))

# Remove empty text
before = len(df)
df = df[df["text_clean"].str.len() > 0]
print("Removed empty text rows:", before - len(df))

print("Rows after cleaning:", len(df))

Dropped duplicates by url: 0
Dropped duplicates by id: 0
Dropped exact duplicate rows: 0
Removed empty text rows: 0
Rows after cleaning: 110


In [9]:
if "publishedAt" in df.columns:
    df["publishedAt"] = pd.to_datetime(df["publishedAt"], errors="coerce", utc=True)
    print("Missing publishedAt:", df["publishedAt"].isna().sum())
else:
    df["publishedAt"] = pd.NaT

Missing publishedAt: 0


In [10]:
def preprocess_advanced(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words and len(t) > 2]

    doc = nlp(" ".join(tokens))
    lemmas = [tok.lemma_ for tok in doc if tok.lemma_ != "-PRON-"]

    return " ".join(lemmas)

In [11]:
nltk.download('punkt_tab')
df["text_processed"] = df["text_clean"].apply(preprocess_advanced)

df[["text_clean", "text_processed"]].head(5)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,text_clean,text_processed
0,Moderna shares drop after FDA refuses to file ...,moderna share drop fda refuse file influenza v...
1,FDA refuses to review Moderna’s mRNA flu vacci...,fda refuse review moderna mrna flu vaccine app...
2,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...
3,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...
4,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...


In [12]:
final_cols = [
    "url",
    "publishedAt",
    "title",
    "description",
    "content",
    "source_name",
    "source_url",
    "text_full",
    "text_clean",
    "text_processed"
]
final_cols = [c for c in final_cols if c in df.columns]

df_struct = df[final_cols].copy()
print("Final structured rows:", len(df_struct))
df_struct.head(5)

Final structured rows: 110


,url,publishedAt,title,description,content,source_name,source_url,text_full,text_clean,text_processed
0,https://seekingalpha.com/news/4550005-moderna-...,2026-02-11 07:54:39+00:00,Moderna shares drop after FDA refuses to file ...,Moderna’s mRNA-1010 flu vaccine application ge...,Moderna shares drop after FDA refuses to file ...,Seeking Alpha,https://seekingalpha.com,Moderna shares drop after FDA refuses to file ...,Moderna shares drop after FDA refuses to file ...,moderna share drop fda refuse file influenza v...
1,https://www.foxnews.com/health/fda-refuses-rev...,2026-02-11 07:20:19+00:00,FDA refuses to review Moderna’s mRNA flu vacci...,FDA refuses Moderna mRNA flu vaccine applicati...,NEWYou can now listen to Fox News articles!\nT...,Fox News,https://www.foxnews.com,FDA refuses to review Moderna’s mRNA flu vacci...,FDA refuses to review Moderna’s mRNA flu vacci...,fda refuse review moderna mrna flu vaccine app...
2,https://www.cbsnews.com/news/moderna-says-fda-...,2026-02-11 04:28:28+00:00,Moderna says FDA refuses its application for n...,The Food and Drug Administration is refusing t...,The Food and Drug Administration is refusing t...,CBS News,https://www.cbsnews.com,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...
3,https://www.cnbctv18.com/lifestyle/healthcare/...,2026-02-11 03:43:35+00:00,Moderna says FDA refuses its application for n...,Moderna faces FDA refusal for its mRNA flu vac...,FDA under Robert F. Kennedy Jr. refused to con...,CNBC TV18,https://www.cnbctv18.com,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...
4,https://www.pennlive.com/health/2026/02/kenned...,2026-02-11 02:55:05+00:00,Moderna says FDA refuses its application for n...,The decision follows a pattern of policy shift...,"By LAURAN NEERGAARD and MATTHEW PERRONE, The A...",Mechanicsburg Patriot News,https://www.pennlive.com,Moderna says FDA refuses its application for n...,Moderna says FDA refuses its application for n...,moderna say fda refuse application new mrna fl...


In [13]:
out_csv = os.path.join(OUT_DIR, "gnews_articles_phase2_cleaned.csv")
df_struct.to_csv(out_csv, index=False)

print("Saved:", out_csv)

Saved: phase2_outputs/gnews_articles_phase2_cleaned.csv


In [16]:
# Vectorization (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib
import os
import numpy as np

# Use processed text for vectorization
texts = df_struct["text_processed"].fillna("").astype(str)


vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(texts)
print("TF-IDF shape:", X.shape)

# Save sparse matrix
tfidf_path = os.path.join(OUT_DIR, "gnews_tfidf_matrix.npz")
save_npz(tfidf_path, X)
print("Saved TF-IDF matrix:", tfidf_path)

# Save the vectorizer
vec_path = os.path.join(OUT_DIR, "gnews_tfidf_vectorizer.pkl")
joblib.dump(vectorizer, vec_path)
print("Saved vectorizer:", vec_path)

# Show top 10 terms by average TF-IDF
avg_tfidf = np.asarray(X.mean(axis=0)).ravel()
top_idx = avg_tfidf.argsort()[::-1][:10]
top_terms = vectorizer.get_feature_names_out()[top_idx]
top_scores = avg_tfidf[top_idx]

print("Top 10 terms (avg TF-IDF):")
for t, s in zip(top_terms, top_scores):
    print(t, round(float(s), 6))

TF-IDF shape: (110, 4399)
Saved TF-IDF matrix: phase2_outputs/gnews_tfidf_matrix.npz
Saved vectorizer: phase2_outputs/gnews_tfidf_vectorizer.pkl
Top 10 terms (avg TF-IDF):
flu 0.059819
vaccine 0.053023
moderna 0.043332
application 0.041978
mrna 0.039582
flu vaccine 0.039166
new 0.039085
refuse 0.037272
fda 0.035304
application new 0.033684


Preprocessing Validation (Pre-EDA Preview)

This step provides a quick frequency check of the processed corpus to verify that tokenization, stopword removal, and lemmatization were applied correctly.

This is not the formal exploratory data analysis phase. Full EDA, including visualization and interpretation, will be conducted separately.

In [17]:
from collections import Counter

# Use processed text (tokenized earlier + stopwords removed + lemmatized)
texts = df_struct["text_processed"].fillna("").astype(str)

# Split into tokens and count
all_tokens = " ".join(texts).split()
word_freq = Counter(all_tokens)

# Show top N terms
TOP_N = 20
top_terms = word_freq.most_common(TOP_N)

print(f"Top {TOP_N} terms by raw count:")
for term, count in top_terms:
    print(term, count)

Top 20 terms by raw count:
flu 248
vaccine 201
char 107
new 94
moderna 90
application 85
influenza 74
mrna 72
fda 71
refuse 71
administration 65
say 63
health 57
food 54
drug 53
get 51
vaccination 46
review 43
shot 41
season 41
